In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer

In [3]:
data = pd.read_csv("final_data.csv")

In [4]:
player1_data = data.sample(1000, random_state=42)
player2_data = data.sample(1000, random_state=42)

In [5]:
trade_data = pd.DataFrame()
trade_data['PTS_diff'] = player1_data['PTS'] - player2_data['PTS']
trade_data['AST_diff'] = player1_data['AST'] - player2_data['AST']
trade_data['REB_diff'] = player1_data['REB'] - player2_data['REB']
trade_data['Efficiency_diff'] = player1_data['Efficiency'] - player2_data['Efficiency']
trade_data['FG_PCT_diff'] = player1_data['FG_PCT'] - player2_data['FG_PCT']
trade_data['FG3_PCT_diff'] = player1_data['FG3_PCT'] - player2_data['FG3_PCT']
trade_data['FT_PCT_diff'] = player1_data['FT_PCT'] - player2_data['FT_PCT']
trade_data['TOV_diff'] = player1_data['TOV'] - player2_data['TOV']
trade_data['STL_diff'] = player1_data['STL_x'] - player2_data['STL_x']
trade_data['BLK_diff'] = player1_data['BLK_x'] - player2_data['BLK_x']
trade_data['MIN_diff'] = player1_data['MIN_x'] - player2_data['MIN_x']

In [6]:
trade_data['trade_outcome'] = np.random.randint(0, 2, trade_data.shape[0])

In [7]:
features = trade_data.drop(columns=['trade_outcome'])
target = trade_data['trade_outcome']

In [8]:
print("Checking for missing values...")
print(features.isna().sum())

Checking for missing values...
PTS_diff           0
AST_diff           0
REB_diff           0
Efficiency_diff    0
FG_PCT_diff        0
FG3_PCT_diff       0
FT_PCT_diff        0
TOV_diff           0
STL_diff           0
BLK_diff           0
MIN_diff           0
dtype: int64


In [9]:
imputer = SimpleImputer(strategy='mean')
features = imputer.fit_transform(features)

In [10]:
features = pd.DataFrame(features, columns=trade_data.drop(columns=['trade_outcome']).columns)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [14]:
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy}')

Test Accuracy: 0.485


In [15]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       103
           1       0.48      1.00      0.65        97

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.33       200
weighted avg       0.24      0.48      0.32       200



/Users/adityarajsingh/miniforge3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/adityarajsingh/miniforge3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/adityarajsingh/miniforge3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [16]:
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [17]:
print("\nFeature Importances:")
print(feature_importance_df)


Feature Importances:
            Feature  Importance
0          PTS_diff         0.0
1          AST_diff         0.0
2          REB_diff         0.0
3   Efficiency_diff         0.0
4       FG_PCT_diff         0.0
5      FG3_PCT_diff         0.0
6       FT_PCT_diff         0.0
7          TOV_diff         0.0
8          STL_diff         0.0
9          BLK_diff         0.0
10         MIN_diff         0.0


In [18]:
player2 = data[data['PLAYER_NAME'] == 'AJ Green'].select_dtypes(include=[np.number]).mean()
player1 = data[data['PLAYER_NAME'] == 'Stephen Curry'].select_dtypes(include=[np.number]).mean()

In [19]:
test_trade = pd.DataFrame({
    'PTS_diff': [player2['PTS'] - player1['PTS']],
    'AST_diff': [player2['AST'] - player1['AST']],
    'REB_diff': [player2['REB'] - player1['REB']],
    'Efficiency_diff': [player2['Efficiency'] - player1['Efficiency']],
    'FG_PCT_diff': [player2['FG_PCT'] - player1['FG_PCT']],
    'FG3_PCT_diff': [player2['FG3_PCT'] - player1['FG3_PCT']],
    'FT_PCT_diff': [player2['FT_PCT'] - player1['FT_PCT']],
    'TOV_diff': [player2['TOV'] - player1['TOV']],
    'STL_diff': [player2['STL_x'] - player1['STL_x']],
    'BLK_diff': [player2['BLK_x'] - player1['BLK_x']],
    'MIN_diff': [player2['MIN_x'] - player1['MIN_x']]
})


In [20]:
test_trade = scaler.transform(test_trade)

In [21]:
trade_prediction = rf_model.predict(test_trade)

In [22]:
if trade_prediction > 0.5:
    print("\nIt's a good trade to take Stephen Curry over Alec Burks.")
else:
    print("\nIt's a bad trade to take Stephen Curry over Alec Burks.")


It's a good trade to take Stephen Curry over Alec Burks.


In [23]:
import pickle

In [24]:
pickle.dump(rf_model,open('model_saved','wb'))